# Section 1: The AI Landscape
## AI for Product Managers

In this lab, you'll practice the **Three Paradigms Framework** — the most important decision tool for any PM working with AI.

---

### The Story

Meta launched **Galactica**, an AI for scientific research. Within **3 days**, it was shut down — it confidently generated fake citations and fabricated papers. $10M+ wasted. The technology worked. The **product decisions** failed.

**95% of AI pilots fail** to deliver P&L impact (MIT 2025). The difference? PM decisions, not technology.

---

In [ ]:
# @title Setup (run this cell first)
# @markdown This cell installs dependencies and sets up the interactive widgets.

!pip install -q plotly ipywidgets

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
import plotly.express as px

print('Ready! Scroll down to start the exercises.')

## Exercise 1: Paradigm Mapping

For each product feature below, select the AI paradigm that fits best.

**Remember the framework:**
- **Classical ML** → Prediction from structured data (rows & columns)
- **LLM** → Text understanding & generation
- **Agent** → Multi-step autonomous tasks with tool use

In [ ]:
# @title Exercise 1: Match Features to Paradigms
# @markdown Select the best paradigm for each feature, then click "Check Answers"

scenarios = [
    {
        'feature': '1. Predict which customers will churn next month based on usage data',
        'correct': 'Classical ML',
        'explanation': 'This is prediction from structured data (usage metrics, login frequency, etc.). Classical ML excels here — it\'s faster, cheaper, and more explainable.'
    },
    {
        'feature': '2. Generate personalized email responses to customer complaints',
        'correct': 'LLM',
        'explanation': 'Text generation from context (the complaint) is pure LLM territory. No structured data to predict from, no multi-step actions needed.'
    },
    {
        'feature': '3. Research competitors and compile a weekly market report',
        'correct': 'Agent',
        'explanation': 'This requires multiple steps: search the web, read articles, extract key info, compare, and compile a report. That\'s an agent workflow with tool use.'
    },
    {
        'feature': '4. Detect fraudulent credit card transactions in real-time',
        'correct': 'Classical ML',
        'explanation': 'Real-time prediction from structured transaction data (amount, location, merchant, time). ML inference runs in milliseconds — perfect for real-time.'
    },
    {
        'feature': '5. Summarize a 50-page contract into key terms and risks',
        'correct': 'LLM',
        'explanation': 'Text understanding and summarization is an LLM\'s sweet spot. Long-context models (Claude 200K, GPT-4 128K) can handle entire contracts.'
    },
    {
        'feature': '6. Process expense reports: scan receipt, categorize, check policy, submit for approval',
        'correct': 'Agent',
        'explanation': 'Multiple steps with different tools: OCR the receipt, categorize the expense, check against policy rules, and submit via an API. Classic agent workflow.'
    }
]

paradigm_options = ['Classical ML', 'LLM', 'Agent']
dropdowns = []
output = widgets.Output()

for s in scenarios:
    label = widgets.HTML(f"<b>{s['feature']}</b>")
    dd = widgets.Dropdown(options=['-- Select --'] + paradigm_options, value='-- Select --',
                          layout=widgets.Layout(width='200px'))
    dropdowns.append(dd)
    display(widgets.HBox([label, dd]))

check_btn = widgets.Button(description='Check Answers', button_style='primary')

def check_answers(b):
    with output:
        clear_output(wait=True)
        score = 0
        html = '<div style="margin-top: 10px;">'
        for i, (dd, s) in enumerate(zip(dropdowns, scenarios)):
            correct = dd.value == s['correct']
            if correct:
                score += 1
            icon = '✅' if correct else '❌'
            color = '#059669' if correct else '#dc2626'
            html += f'<div style="padding: 8px; margin: 4px 0; background: {"#ecfdf5" if correct else "#fef2f2"}; border-radius: 6px;">'
            html += f'{icon} <b>Q{i+1}:</b> You said <b>{dd.value}</b> — correct answer is <b>{s["correct"]}</b><br>'
            html += f'<span style="color: #6b7280; font-size: 0.9em;">{s["explanation"]}</span></div>'
        html += f'<h3 style="margin-top: 16px;">Score: {score}/{len(scenarios)}</h3>'
        if score == len(scenarios):
            html += '<p style="color: #059669;">Perfect! You\'ve mastered the Three Paradigms Framework.</p>'
        elif score >= 4:
            html += '<p style="color: #d97706;">Great job! Review the ones you missed — the explanations highlight the key decision factors.</p>'
        else:
            html += '<p style="color: #dc2626;">Review the framework: Structured data prediction → ML, Text tasks → LLM, Multi-step + tools → Agent</p>'
        html += '</div>'
        display(HTML(html))

check_btn.on_click(check_answers)
display(check_btn)
display(output)

## Exercise 2: Paradigm Comparison Radar Chart

Adjust the sliders to see how different paradigms compare across key dimensions.

This helps you visualize the **tradeoffs** when choosing an approach.

In [ ]:
# @title Exercise 2: Interactive Paradigm Comparison
# @markdown Adjust the importance of each factor to see which paradigm fits your use case best.

# Paradigm scores (1-5) on each dimension
paradigm_profiles = {
    'Classical ML': {'Speed to POC': 2, 'Ongoing Cost': 5, 'Explainability': 5, 'Flexibility': 2, 'Data Efficiency': 1, 'Autonomy': 1},
    'LLM': {'Speed to POC': 5, 'Ongoing Cost': 3, 'Explainability': 2, 'Flexibility': 5, 'Data Efficiency': 5, 'Autonomy': 2},
    'Agent': {'Speed to POC': 3, 'Ongoing Cost': 2, 'Explainability': 3, 'Flexibility': 4, 'Data Efficiency': 4, 'Autonomy': 5}
}

dimensions = list(paradigm_profiles['Classical ML'].keys())

weight_sliders = {}
for dim in dimensions:
    weight_sliders[dim] = widgets.FloatSlider(
        value=3, min=1, max=5, step=0.5,
        description=f'{dim}:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='400px')
    )

radar_output = widgets.Output()

def update_radar(*args):
    with radar_output:
        clear_output(wait=True)
        weights = {dim: weight_sliders[dim].value for dim in dimensions}
        
        fig = go.Figure()
        colors = {'Classical ML': '#1e40af', 'LLM': '#d97706', 'Agent': '#059669'}
        
        scores = {}
        for name, profile in paradigm_profiles.items():
            weighted = [profile[d] * weights[d] / 5 for d in dimensions]
            scores[name] = sum(weighted)
            vals = [profile[d] for d in dimensions] + [profile[dimensions[0]]]
            cats = dimensions + [dimensions[0]]
            fig.add_trace(go.Scatterpolar(
                r=vals, theta=cats, fill='toself',
                name=name, line=dict(color=colors[name]),
                opacity=0.6
            ))
        
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
            showlegend=True, title='Paradigm Comparison (raw scores)',
            template='plotly_white', height=450
        )
        fig.show()
        
        winner = max(scores, key=scores.get)
        display(HTML(f'<div style="background:#eff6ff; padding:12px; border-radius:8px; margin-top:8px;">'
                     f'<b>Based on your priorities:</b> {winner} scores highest '
                     f'({scores[winner]:.1f}) — '
                     f'ML: {scores["Classical ML"]:.1f} | LLM: {scores["LLM"]:.1f} | Agent: {scores["Agent"]:.1f}'
                     f'</div>'))

for s in weight_sliders.values():
    s.observe(update_radar, names='value')

display(widgets.HTML('<b>How important is each factor for YOUR use case? (1=low, 5=critical)</b>'))
for s in weight_sliders.values():
    display(s)
display(radar_output)
update_radar()

## Discussion Prompts

Talk with your table group:

1. **Think of an AI feature your company has launched or considered.** Which paradigm was used? Was it the right choice?

2. **What's the riskiest paradigm mistake you've seen?** (Using LLM when ML was better, or vice versa)

3. **"We should just use ChatGPT for everything."** How would you respond to this from a stakeholder?

---

## Stakeholder Framing

> *"AI isn't one thing — it's three different approaches, each with different strengths. The companies in the successful 5% match the tool to the job. That's what we'll do."*

---

### Interactive Tools (Open in New Tab)
- [AI Paradigm Picker](https://huggingface.co/spaces/axelsirota/ai-paradigm-picker) — Describe a feature, get a paradigm recommendation
- [Model Comparison Arena](https://huggingface.co/spaces/axelsirota/model-comparison-arena) — Compare LLM responses side by side
- [Build vs Buy Calculator](https://huggingface.co/spaces/axelsirota/build-vs-buy-calculator) — TCO analysis for AI projects